In [ ]:
# Note: Download base_name_test_fold.txt from the github before running the inference code.

In [29]:
# Clone the main repository  (Run thses cells in sequence)
!git clone https://github.com/limhoyeon/ToothGroupNetwork.git

In [29]:
!pip install wandb
!pip install --ignore-installed PyYAML
!pip install open3d
!pip install multimethod
!pip install termcolor
!pip install trimesh
!pip install easydict

In [ ]:
#===============================================================================
#    Modifies the codes to to make it compatible with current version (Start)
#===============================================================================

In [4]:
# since there is not THC in recent version of PyTorch (specifically since version 1.11).
# Changing the source code to make it compatible

import os

# Define the path to the target file
file_path = '/content/ToothGroupNetwork/external_libs/pointops/src/aggregation/aggregation_cuda.cpp'

# Define the new content
new_content = """#include <vector>
// REMOVED: #include <THC/THC.h>
#include <torch/serialize/tensor.h>
#include <ATen/cuda/CUDAContext.h>
#include <torch/extension.h> // ADDED: Required for modern PyTorch C++ extensions
#include "aggregation_cuda_kernel.h"

void aggregation_forward_cuda(int n, int nsample, int c, int w_c, at::Tensor input_tensor, at::Tensor position_tensor, at::Tensor weight_tensor, at::Tensor idx_tensor, at::Tensor output_tensor)
{
    const float *input = input_tensor.data_ptr<float>();
    const float *position = position_tensor.data_ptr<float>();
    const float *weight = weight_tensor.data_ptr<float>();
    const int *idx = idx_tensor.data_ptr<int>();
    float *output = output_tensor.data_ptr<float>();
    aggregation_forward_cuda_launcher(n, nsample, c, w_c, input, position, weight, idx, output);
}

void aggregation_backward_cuda(int n, int nsample, int c, int w_c, at::Tensor input_tensor, at::Tensor position_tensor, at::Tensor weight_tensor, at::Tensor idx_tensor, at::Tensor grad_output_tensor, at::Tensor grad_input_tensor, at::Tensor grad_position_tensor, at::Tensor grad_weight_tensor)
{
	const float *input = input_tensor.data_ptr<float>();
    const float *position = position_tensor.data_ptr<float>();
    const float *weight = weight_tensor.data_ptr<float>();
    const int *idx = idx_tensor.data_ptr<int>();
    const float *grad_output = grad_output_tensor.data_ptr<float>();
    float *grad_input = grad_input_tensor.data_ptr<float>();
    float *grad_position = grad_position_tensor.data_ptr<float>();
    float *grad_weight = grad_weight_tensor.data_ptr<float>();
    aggregation_backward_cuda_launcher(n, nsample, c, w_c, input, position, weight, idx, grad_output, grad_input, grad_position, grad_weight);
}
"""

# Check if the file exists before writing
if os.path.exists(file_path):
    try:
        with open(file_path, 'w') as f:
            f.write(new_content)
        print(f"Successfully updated file: {file_path}")
    except IOError as e:
        print(f"Error writing to file {file_path}: {e}")
else:
    print(f"Error: File not found at path: {file_path}")

Successfully updated file: /content/ToothGroupNetwork/external_libs/pointops/src/aggregation/aggregation_cuda.cpp


In [5]:
import os

def override_specific_file_content():
    file_path = '/content/ToothGroupNetwork/external_libs/pointops/src/grouping/grouping_cuda.cpp'

    if not os.path.exists(file_path):
        print(f"Error: Could not find the file at {file_path}")
        return

    new_content = """#include <vector>
#include <torch/serialize/tensor.h>
#include <ATen/cuda/CUDAContext.h>
#include <torch/extension.h> // Use modern PyTorch extension API
#include "grouping_cuda_kernel.h"


void grouping_forward_cuda(int m, int nsample, int c, at::Tensor input_tensor, at::Tensor idx_tensor, at::Tensor output_tensor)
{
    const float *input = input_tensor.data_ptr<float>();
    const int *idx = idx_tensor.data_ptr<int>();
    float *output = output_tensor.data_ptr<float>();
    grouping_forward_cuda_launcher(m, nsample, c, input, idx, output);
}

void grouping_backward_cuda(int m, int nsample, int c, at::Tensor grad_output_tensor, at::Tensor idx_tensor, at::Tensor grad_input_tensor)
{
    const float *grad_output = grad_output_tensor.data_ptr<float>();
    const int *idx = idx_tensor.data_ptr<int>();
    float *grad_input = grad_input_tensor.data_ptr<float>();
    grouping_backward_cuda_launcher(m, nsample, c, grad_output, idx, grad_input);
}
"""
    try:
        with open(file_path, 'w') as f:
            f.write(new_content)
        print(f"Successfully updated {file_path} to use modern PyTorch headers.")
        print("You can now attempt to rebuild the extension.")

    except IOError as e:
        print(f"Failed to write to file {file_path}: {e}")

if __name__ == "__main__":
    override_specific_file_content()



Successfully updated /content/ToothGroupNetwork/external_libs/pointops/src/grouping/grouping_cuda.cpp to use modern PyTorch headers.
You can now attempt to rebuild the extension.


In [6]:
import os

def override_specific_file_content():
    file_path = '/content/ToothGroupNetwork/external_libs/pointops/src/interpolation/interpolation_cuda.cpp'

    if not os.path.exists(file_path):
        print(f"Error: Could not find the file at {file_path}")
        return

    new_content = """#include <vector>
#include <torch/serialize/tensor.h>
#include <ATen/cuda/CUDAContext.h>
#include <torch/extension.h> // Use modern PyTorch extension API
#include "interpolation_cuda_kernel.h"


void interpolation_forward_cuda(int n, int c, int k, at::Tensor input_tensor, at::Tensor idx_tensor, at::Tensor weight_tensor, at::Tensor output_tensor)
{
    const float *input = input_tensor.data_ptr<float>();
    const int *idx = idx_tensor.data_ptr<int>();
    const float *weight = weight_tensor.data_ptr<float>();
    float *output = output_tensor.data_ptr<float>();
    interpolation_forward_cuda_launcher(n, c, k, input, idx, weight, output);
}

void interpolation_backward_cuda(int n, int c, int k, at::Tensor grad_output_tensor, at::Tensor idx_tensor, at::Tensor weight_tensor, at::Tensor grad_input_tensor)
{
    const float *grad_output = grad_output_tensor.data_ptr<float>();
    const int *idx = idx_tensor.data_ptr<int>();
    const float *weight = weight_tensor.data_ptr<float>();
    float *grad_input = grad_input_tensor.data_ptr<float>();
    interpolation_backward_cuda_launcher(n, c, k, grad_output, idx, weight, grad_input);
}
"""
    try:
        with open(file_path, 'w') as f:
            f.write(new_content)
        print(f"Successfully updated {file_path} to use modern PyTorch headers.")
        print("You may need to run this for all problematic files before rebuilding the extension.")

    except IOError as e:
        print(f"Failed to write to file {file_path}: {e}")

if __name__ == "__main__":
    override_specific_file_content()


Successfully updated /content/ToothGroupNetwork/external_libs/pointops/src/interpolation/interpolation_cuda.cpp to use modern PyTorch headers.
You may need to run this for all problematic files before rebuilding the extension.


In [7]:
import os

def override_specific_file_content():
    file_path = '/content/ToothGroupNetwork/external_libs/pointops/src/knnquery/knnquery_cuda.cpp'

    if not os.path.exists(file_path):
        print(f"Error: Could not find the file at {file_path}")
        return

    new_content = """#include <vector>
#include <torch/serialize/tensor.h>
#include <ATen/cuda/CUDAContext.h>
#include <torch/extension.h> // Use modern PyTorch extension API
#include "knnquery_cuda_kernel.h"


void knnquery_cuda(int m, int nsample, at::Tensor xyz_tensor, at::Tensor new_xyz_tensor, at::Tensor offset_tensor, at::Tensor new_offset_tensor, at::Tensor idx_tensor, at::Tensor dist2_tensor)
{
    const float *xyz = xyz_tensor.data_ptr<float>();
    const float *new_xyz = new_xyz_tensor.data_ptr<float>();
    const int *offset = offset_tensor.data_ptr<int>();
    const int *new_offset = new_offset_tensor.data_ptr<int>();
    int *idx = idx_tensor.data_ptr<int>();
    float *dist2 = dist2_tensor.data_ptr<float>();
    knnquery_cuda_launcher(m, nsample, xyz, new_xyz, offset, new_offset, idx, dist2);
}
"""
    try:
        with open(file_path, 'w') as f:
            f.write(new_content)
        print(f"Successfully updated {file_path} to use modern PyTorch headers.")
        print("Run this script for any other files with the THC error before rebuilding.")

    except IOError as e:
        print(f"Failed to write to file {file_path}: {e}")

if __name__ == "__main__":
    override_specific_file_content()


Successfully updated /content/ToothGroupNetwork/external_libs/pointops/src/knnquery/knnquery_cuda.cpp to use modern PyTorch headers.
Run this script for any other files with the THC error before rebuilding.


In [8]:
import os

def override_specific_file_content():
    file_path = '/content/ToothGroupNetwork/external_libs/pointops/src/sampling/sampling_cuda.cpp'

    if not os.path.exists(file_path):
        print(f"Error: Could not find the file at {file_path}")
        return

    new_content = """#include <vector>
#include <torch/serialize/tensor.h>
#include <ATen/cuda/CUDAContext.h>
#include <torch/extension.h> // Use modern PyTorch extension API
#include "sampling_cuda_kernel.h"


void furthestsampling_cuda(int b, int n, at::Tensor xyz_tensor, at::Tensor offset_tensor, at::Tensor new_offset_tensor, at::Tensor tmp_tensor, at::Tensor idx_tensor)
{
    const float *xyz = xyz_tensor.data_ptr<float>();
    const int *offset = offset_tensor.data_ptr<int>();
    const int *new_offset = new_offset_tensor.data_ptr<int>();
    float *tmp = tmp_tensor.data_ptr<float>();
    int *idx = idx_tensor.data_ptr<int>();
    furthestsampling_cuda_launcher(b, n, xyz, offset, new_offset, tmp, idx);
}
"""
    try:
        with open(file_path, 'w') as f:
            f.write(new_content)
        print(f"Successfully updated {file_path} to use modern PyTorch headers.")
        print("Run this script for any remaining files with the THC error before rebuilding.")

    except IOError as e:
        print(f"Failed to write to file {file_path}: {e}")

if __name__ == "__main__":
    override_specific_file_content()


Successfully updated /content/ToothGroupNetwork/external_libs/pointops/src/sampling/sampling_cuda.cpp to use modern PyTorch headers.
Run this script for any remaining files with the THC error before rebuilding.


In [9]:
import os

def override_specific_file_content():
    file_path = '/content/ToothGroupNetwork/external_libs/pointops/src/subtraction/subtraction_cuda.cpp'

    if not os.path.exists(file_path):
        print(f"Error: Could not find the file at {file_path}")
        return

    new_content = """#include <vector>
#include <torch/serialize/tensor.h>
#include <ATen/cuda/CUDAContext.h>
#include <torch/extension.h> // Use modern PyTorch extension API
#include "subtraction_cuda_kernel.h"


void subtraction_forward_cuda(int n, int nsample, int c, at::Tensor input1_tensor, at::Tensor input2_tensor, at::Tensor idx_tensor, at::Tensor output_tensor)
{
    const float *input1 = input1_tensor.data_ptr<float>();
    const float *input2 = input2_tensor.data_ptr<float>();
    const int *idx = idx_tensor.data_ptr<int>();
    float *output = output_tensor.data_ptr<float>();
    subtraction_forward_cuda_launcher(n, nsample, c, input1, input2, idx, output);
}

void subtraction_backward_cuda(int n, int nsample, int c, at::Tensor idx_tensor, at::Tensor grad_output_tensor, at::Tensor grad_input1_tensor, at::Tensor grad_input2_tensor)
{
    const int *idx = idx_tensor.data_ptr<int>();
    const float *grad_output = grad_output_tensor.data_ptr<float>();
    float *grad_input1 = grad_input1_tensor.data_ptr<float>();
    float *grad_input2 = grad_input2_tensor.data_ptr<float>();
    subtraction_backward_cuda_launcher(n, nsample, c, idx, grad_output, grad_input1, grad_input2);
}
"""
    try:
        with open(file_path, 'w') as f:
            f.write(new_content)
        print(f"Successfully updated {file_path} to use modern PyTorch headers.")
        print("All specified files should now be updated. Please attempt to rebuild the extension.")

    except IOError as e:
        print(f"Failed to write to file {file_path}: {e}")

if __name__ == "__main__":
    override_specific_file_content()


Successfully updated /content/ToothGroupNetwork/external_libs/pointops/src/subtraction/subtraction_cuda.cpp to use modern PyTorch headers.
All specified files should now be updated. Please attempt to rebuild the extension.


In [ ]:
#===============================================================================
#                          End of the modification
#===============================================================================

In [10]:
# 1. Change your current directory to the project root
!cd /content/ToothGroupNetwork/

In [29]:
!pip install gdown

In [29]:
# Downloads ckpts(new).zip that contains different  models checkpoints.

import gdown

# Your normal Google Drive link
url = "https://drive.google.com/file/d/1Xpw9vYbDVwoafVohqY0aMen2iMCx053-/view?usp=drive_link"

# gdown can automatically figure out the file ID and original filename
gdown.download(url, quiet=False, fuzzy=True)

In [ ]:
# ***************************************************************************************************************************************

# *****************    Codes for downloading and managing the input structure for the inference  ****************************************



In [ ]:
!cd '/content/ToothGroupNetwork'

In [14]:
import os
import shutil
from zipfile import ZipFile
# https://drive.google.com/file/d/140qWvJR3zV-6j-A2uzi1vBfPa9M87HaA/view?usp=drive_link
# https://drive.google.com/file/d/1N_ZewZ1yUTDKo3fe7ZLGgUho8_iWrFbi/view?usp=drive_link
# https://drive.google.com/file/d/1GKjz4YaoXlXwsvPCVKXT07-AstLQ6u-y/view?usp=drive_link
# https://drive.google.com/file/d/1x9Euejg6RL_9HXgdNYmqRtV2Em6YuFtr/view?usp=drive_link
# Define the Google Drive file IDs and desired local filenames (placeholders)
# Replace 'YOUR_FILE_ID_HERE' with the actual IDs from the challenge links.
# These IDs link to specific files on Google Drive that must be made public.
obj_zip_1_id = '140qWvJR3zV-6j-A2uzi1vBfPa9M87HaA'
obj_zip_2_id = '1N_ZewZ1yUTDKo3fe7ZLGgUho8_iWrFbi'
json_zip_1_id = '1GKjz4YaoXlXwsvPCVKXT07-AstLQ6u-y'
json_zip_2_id = '1x9Euejg6RL_9HXgdNYmqRtV2Em6YuFtr'

obj_zip_1_name = '3D_scans_per_patient_obj_files.zip'
obj_zip_2_name = '3D_scans_per_patient_obj_files_b2.zip'
json_zip_1_name = 'ground-truth_labels_instances.zip'
json_zip_2_name = 'ground-truth_labels_instances_b2.zip'

# Define final parent directories where all data will reside
data_obj_parent_directory = 'data_obj_parent_directory'
data_json_parent_directory = 'data_json_parent_directory'

# Define temporary folders for unzipping files initially
temp_obj_1 = 'temp_obj_1'
temp_obj_2 = 'temp_obj_2'
temp_json_1 = 'temp_json_1'
temp_json_2 = 'temp_json_2'


In [15]:
# Function to download files from Google Drive using gdown
# Make sure the files are publicly accessible ("Anyone with the link")
def download_from_drive(file_id, output_path):
    print(f"Downloading {output_path}...")
    # 'gdown' is a command-line utility preinstalled in Colab environments
    os.system(f'gdown --id {file_id} -O {output_path}')
    print("Download complete.")

# Function to unzip files to a specified target directory
def unzip_file(zip_name, extract_path):
    print(f"Extracting {zip_name} to {extract_path}...")
    os.makedirs(extract_path, exist_ok=True) # Ensure the destination directory exists
    with ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall(extract_path) # Extract all contents to the specified path
    print("Extraction complete.")

# Function to merge contents of a source folder into a destination folder
def merge_folders(source_folder, destination_folder):
    print(f"Merging contents of {source_folder} into {destination_folder}...")
    for item in os.listdir(source_folder):
        s = os.path.join(source_folder, item)
        d = os.path.join(destination_folder, item)
        if os.path.isdir(s):
            # Recursively copy directories, allowing existing dirs
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d) # Copy single files
    print("Merge complete.")


In [29]:
# Create the final destination directories
os.makedirs(data_obj_parent_directory, exist_ok=True)
os.makedirs(data_json_parent_directory, exist_ok=True)

# --- Download Step ---
# You need to uncomment and run these lines after replacing 'YOUR_FILE_ID_HERE'
download_from_drive(obj_zip_1_id, obj_zip_1_name)
download_from_drive(obj_zip_2_id, obj_zip_2_name)
download_from_drive(json_zip_1_id, json_zip_1_name)
download_from_drive(json_zip_2_id, json_zip_2_name)

# --- Unzip Step ---
# Assumes the zip files are present in the current working directory after download
unzip_file(obj_zip_1_name, temp_obj_1)
unzip_file(obj_zip_2_name, temp_obj_2)
unzip_file(json_zip_1_name, temp_json_1)
unzip_file(json_zip_2_name, temp_json_2)

# --- Merge Step ---
# NOTE: You might need to adjust the source path (e.g., os.path.join(temp_obj_1, 'some_subdir'))
# depending on how the original zip files are structured. The current code assumes
# the patient folders (like '000MSZGW') are in the root of the extracted temporary folders.
merge_folders(temp_obj_1, data_obj_parent_directory)
merge_folders(temp_obj_2, data_obj_parent_directory)
merge_folders(temp_json_1, data_json_parent_directory)
merge_folders(temp_json_2, data_json_parent_directory)

print("\nTemporary merging steps complete. Proceeding to cleanup.")


In [29]:
# Clean up temporary folders
shutil.rmtree(temp_obj_1)
shutil.rmtree(temp_obj_2)
shutil.rmtree(temp_json_1)
shutil.rmtree(temp_json_2)

# Clean up the original zip files (uncomment if desired)
# os.remove(obj_zip_1_name)
# os.remove(obj_zip_2_name)
# os.remove(json_zip_1_name)
# os.remove(json_zip_2_name)

print("\nData organization complete. Temporary files cleaned up.")
print(f"Your final data is located in: {data_obj_parent_directory} and {data_json_parent_directory}")


In [29]:
%cd /content/ToothGroupNetwork

In [29]:
!cd external_libs/pointops && python setup.py install

In [29]:
# Moving to the ToothGroupNetwork Directory
!mv /content/data_json_parent_directory /content/data_obj_parent_directory   /content/ToothGroupNetwork


In [24]:
!mkdir ckpts

In [29]:
# Unzipping ckptd(new).zip to the ckps folder inside ToothGroupNetwork directory

!unzip "/content/ckpts(new).zip" -d /content/ToothGroupNetwork/ckpts

In [29]:

#****************************************************************
#         Final Inference Code
#****************************************************************

!python start_inference.py \
--input_dir_path /content/ToothGroupNetwork/data_obj_parent_directory \
--split_txt_path /content/base_name_test_fold.txt \
--save_path /content/ToothGroupNetwork/output_directory_pointtransformer \
--model_name pointtransformer \
--checkpoint_path /content/ToothGroupNetwork/ckpts/pointtransformer      # Note: Here,you should use the full path without .h5 extension
